# VISCERA — concept-supervised method (full pipeline, A6000/A100)
**VIS**ual-**C**oncept **E**ndoscopy **R**epresentation via **A**ttributes. Pipeline: SSL weight (dinov2.pth) → **Stage-1: concept-supervised pretrain on ~160k (35 clinical concepts)** with **L2-SP anchor** (keep SSL features) + GRL center-invariance → **Stage-2: downstream neo/ndbe** (PPV@90R tail loss + WiSE-FT + multi-seed ensemble).

**Drive `DRIVE_DIR` needs:** numbered data zips `0.zip..N.zip` (→ out/<dir>), `train.zip`/`val.zip` (→ out/train, out/val), `dinov2.pth`. No dataset.zip needed (labels come from the JSONs). Runtime → **GPU**.

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
REPO_URL = 'https://github.com/HuynhDoTanThanh/RARE2026.git'   # <-- your repo
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

In [ ]:
# ---- extract data from Drive (numbered chunks -> out/<dir>; train/val.zip -> out/train,out/val) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- your Drive folder
import glob, os, zipfile, shutil
assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
os.makedirs('out', exist_ok=True)
def extract_chunk(zpath):
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        target = f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ('images' in tops or 'labels' in tops) else ('.' if tops=={'out'} else 'out')
        os.makedirs(target, exist_ok=True); zf.extractall(target)
for z in [p for p in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(p).lower()]:
    extract_chunk(z); print('extracted', os.path.basename(z))
print('out dirs:', len(glob.glob('out/*/labels')), '| train labels:', len(glob.glob('out/train/labels/*.json')), '| train imgs:', len(glob.glob('out/train/images/*')))

In [ ]:
# ---- labeled CSVs from the JSON labels (img = out/<split>/images/<name>.jpg) ----
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

In [ ]:
# ---- 35-concept target matrix (needed only for Stage-1 pretraining + CTM mining). Reuse from Drive if cached. ----
import os, shutil
os.makedirs('phase3/cache', exist_ok=True)
if os.path.exists('phase3/cache/concept_targets.npz'):
    print('concept_targets.npz already present locally.')
elif os.path.exists(f'{DRIVE_DIR}/concept_targets.npz'):
    shutil.copy(f'{DRIVE_DIR}/concept_targets.npz', 'phase3/cache/concept_targets.npz')
    print('REUSED concept_targets.npz from Drive.')
else:
    !python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz
    shutil.copy('phase3/cache/concept_targets.npz', f'{DRIVE_DIR}/concept_targets.npz')
    print('built + cached concept_targets.npz to Drive.')

## Stage-1 — concept-supervised pretraining (full-35 losses: certain / uncertain / smoothing)

LIGHT + L2-SP anchor → *shape*, don't overwrite SSL. Upgrades vs the old masked-BCE:
- **`--discrim full15`** — the full discriminative core (9 → 15; adds `color_heterogeneity, whitish_focal_area, vascular_irregularity, dilated_vessels, focal_abnormal_vessels, border_sharpness`). This is the real substance of "more labels".
- **certain** = class-balanced soft-BCE (`--pos_weight_cap 10` rebalances rare-positive concepts, e.g. `depression_ulceration` p=.055 → 10× — the term most likely to move PPV@90R).
- **uncertain** (`--unc_w 0.1`) = prior-pull on `not_assessable` cells instead of hard-masking (never invents a label).
- **smoothing** (`--smooth_eps 0.05`) = target shrinks toward the per-concept prior so the head can't get more confident than the noisy VLM warrants (anti center-memorization).

Every knob **defaults to the old masked-BCE** → clean superset, ablatable via the flags. Note: **7 of the 35 concepts are dead constants** (value ≡ 0: modality/distance/view/landmark/interpretable_fraction/dominant_color/lesion_size) and are auto-dropped; context/acquisition concepts go to a **detached AUX head** so "all 35" never re-injects center style into the trunk.

In [ ]:
# Stage-1: concept-supervised pretraining -> concept_encoder.pt.
# REUSE it if a concept_encoder.pt already exists on Drive (Stage-1 is the slow part; skip retraining).
import os, shutil
if os.path.exists(f'{DRIVE_DIR}/concept_encoder.pt'):
    shutil.copy(f'{DRIVE_DIR}/concept_encoder.pt', 'concept_encoder.pt')
    print('REUSED concept_encoder.pt from Drive -> Stage-1 SKIPPED (delete it on Drive to re-pretrain).')
else:
    print('no concept_encoder.pt on Drive -> running Stage-1 pretraining (~15 epochs) ...')
    # full discriminative core + certain/uncertain/smoothing losses (knobs default OFF == old masked-BCE).
    !python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz \
        --unfreeze 3 --epochs 15 --bs 192 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 \
        --discrim full15 --context_route detach \
        --certain_floor 0.7 --smooth_eps 0.05 --unc_w 0.1 --pos_weight_cap 10 \
        --out concept_encoder.pt
    shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt')   # cache to Drive for next runs
    print('saved concept_encoder.pt to Drive.')

## Stage-2 — downstream from the concept encoder → 3-seed ship

Recipe = **fine-tune last-6 blocks + WiSE-FT 0.7** (the config behind exp1 = leaderboard PPV@90R **0.0181**). WiSE-FT interpolates the fine-tuned backbone back toward the SSL init, which **recovers the out-of-distribution robustness of linear-probing** (Kumar 2022: full FT distorts pretrained features under shift; Wortsman 2022 WiSE-FT is the fix) while keeping FT's capacity. So the old *head-only vs fine-tune* A/B was just the two ends of one knob — **`--head-only` ≈ `--wise-ft 0`** (frozen backbone, head only). If you want the LP↔FT tradeoff, sweep `--wise-ft {0.3,0.5,0.7,0.9}`, not a separate code path.

Ship trains on **both centers** (`--holdout none`), 3 seeds → ensemble. ⚠️ **New-center validation is deliberately NOT in this path** (the shipped model uses all data). Decide levers on an *external new-center proxy* (**RARE25-val**, the set the platform reports each submission) — the same-center val cell below is optimistic (it said 0.65 vs the real 0.018).

In [ ]:
# ---- LOCO GATE (OPTIONAL) — test the two open recipe questions on the new-center proxy ----
#   Q1 [epochs] does SHORT (12ep) beat EXP1 (30ep)?           (the overfit fix the last gate pointed to)
#   Q2 [SEMI]   does the SEMI loss over the full VLM pool help SHORT?  (Mean-Teacher consistency + one-sided-PU;
#               --semi-steps sweeps the pool with fresh strong-aug, regularizing instead of memorizing 127 pos)
RUN_GATE = False
if RUN_GATE:
    import os, shutil
    os.makedirs('phase3/cache', exist_ok=True)
    if not os.path.exists('phase3/cache/unl_manifest.npz'):                       # unl_manifest.npz = the ONLY thing SEMI needs
        if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
            shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz')
        else:
            !python -m phase3.mine_hardneg --manifest-only
            shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz')
    EXP1  = '--unfreeze 6 --wise-ft 0.7 --epochs 30 --bs 96 --loss bce+rank+pauc --warmup 2'
    SHORT = '--unfreeze 6 --wise-ft 0.7 --epochs 12 --bs 96 --loss bce+rank+pauc --warmup 2'
    SEMI  = SHORT + (' --semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
                     '--semi-n 300000 --semi-bs 256 --semi-steps 10')     # same semi config as the ship cell (94GB)
    LEGS = {'exp1': ('--init concept_encoder.pt', EXP1),
            'short': ('--init concept_encoder.pt', SHORT),
            'semi': ('--init concept_encoder.pt', SEMI)}
    for hold in ['center_2', 'center_1']:
        print(f'\n================ LOCO holdout={hold} ================')
        for name, (ini, flags) in LEGS.items():
            print(f'--- {name} ---')
            !python -m phase3.finetune --train-csv train_colab.csv {ini} --seed 0 --holdout {hold} {flags} --out loco_{name}_{hold}.pt
    import numpy as np, phase3.evaluate as ev
    def leg(hold, name):
        d = np.load(f'loco_{name}_{hold}_loco.npz', allow_pickle=True); return d['y'], d['c'], d['s']
    def cmp(A, B):
        fails = []
        for hold in ['center_2', 'center_1']:
            y, c, sa = leg(hold, A); _, _, sb = leg(hold, B)
            print(f"  holdout {hold}  ({A} vs {B}):")
            for m in ('auroc', 'auprc'):
                g = ev.gate(y, sa, sb, center=c, metric=m, B=2000)
                if g['verdict'] == 'FAIL': fails.append((hold, m))
                print(f"    {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
        return fails
    print('\n===== Q1 [epochs]  SHORT(12ep) vs EXP1(30ep) ====='); f1 = cmp('short', 'exp1')
    print('\n===== Q2 [SEMI]    SEMI vs SHORT ====='); f2 = cmp('semi', 'short')
    print('\n================ DECISIONS ================')
    print('  epochs:', f'SHORT FAILs {f1} -> keep --epochs 30' if f1 else 'ship --epochs 12 (anti-overfit)')
    print('  SEMI:  ', f'SEMI FAILs {f2} -> ship WITHOUT --semi-manifest' if f2 else 'keep SEMI (uses the VLM pool, no regression)')
else:
    print('LOCO gate SKIPPED. Set RUN_GATE=True to test 12ep-vs-30ep AND the SEMI loss (6 runs).')

In [ ]:
# ---- HARD-NEG MINING (OPTIONAL add-on) -> builds phase3/cache/unl_mined.txt ----
# OFF by default: mining ADDS negatives -> MORE positive oversampling -> more overfitting (the exact failure the
# gate exposed at 30 epochs). Only enable AFTER the 12-epoch recipe is settled, and re-gate before shipping.
# One-sided PU: CTM concept-confounded (offline, decisive-ceiling guard) UNION model-in-the-loop FPs.
RUN_MINING = False
if RUN_MINING:
    !python -m phase3.mine_hardneg --confneg-sample 30000 --suspicious-top 5000
    !python -m phase3.mine_hardneg --concept-rank phase3/cache/concept_targets.npz --topn 4000 --skip-top 200
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none --init concept_encoder.pt --seed 0 \
        --unfreeze 6 --wise-ft 0.7 --epochs 12 --out ship_r0.pt
    !python -m phase3.mine_hardneg --score-with ship_r0.pt --topn 4000 --skip-top 200
    import os
    rd = lambda p: [l.strip() for l in open(p) if l.strip()] if os.path.exists(p) else []
    ctm, mfp = rd('phase3/cache/unl_conceptFP.txt'), rd('phase3/cache/unl_modelFP.txt')
    mined = sorted(set(ctm) | set(mfp))
    open('phase3/cache/unl_mined.txt', 'w').write('\n'.join(mined) + '\n')
    print(f'MINED: CTM={len(ctm)} + model-FP={len(mfp)} -> {len(mined)} unique -> phase3/cache/unl_mined.txt')
else:
    print('MINING SKIPPED (RUN_MINING=False). Ship uses labeled negatives only (matches the exp1 baseline).')

In [ ]:
# ship: 3 seeds -> ensemble. RECOMMENDED recipe = SHORT (anti-overfit) + SEMI loss over the FULL VLM pool.
# SEMI: Mean-Teacher consistency on weak/strong-aug unlabeled views + one-sided-PU confident-NEGATIVE (suspicion<0.15).
# 94GB VRAM config: --semi-bs 256 (big batch = VRAM's job) + --semi-steps 10 (grad-accum, ~free VRAM = time's job).
# Coverage/epoch = 26*10*256 ~= 67k -> ~800k frame-draws over 12 epochs = sweeps the WHOLE ~288k pool ~2.8x with
# fresh strong-aug each. To cover more/repeat more, raise --semi-steps (VRAM-free) or --semi-bs (uses VRAM).
import os, shutil
if not os.path.exists('phase3/cache/unl_manifest.npz'):
    if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
        os.makedirs('phase3/cache', exist_ok=True)
        shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz'); print('REUSED unl_manifest.npz from Drive')
    else:
        !python -m phase3.mine_hardneg --manifest-only
        shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz'); print('built + cached unl_manifest.npz')
STAGE2_FLAGS = ('--unfreeze 6 --wise-ft 0.7 --epochs 12 '
                '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
                '--semi-n 300000 --semi-bs 256 --semi-steps 10 --ema-decay 0.99')
# Fallbacks if SEMI regresses on the gate: no-semi '--unfreeze 6 --wise-ft 0.7 --epochs 12' ; exp1 '... --epochs 30'
for s in [0, 1, 2]:
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none --init concept_encoder.pt --seed {s} \
        {STAGE2_FLAGS} --bs 96 --loss bce+rank+pauc --warmup 2 --out ship_seed{s}.pt
shutil.copy('concept_encoder.pt', f'{DRIVE_DIR}/concept_encoder.pt')
for s in [0, 1, 2]: shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')
print('saved encoder + 3 ensemble members to Drive')

## Inference (offline, per-image, ensemble + hflip-TTA)
```
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --images-dir <TEST_DIR> --out preds.csv
```
Knobs to tune: Stage-1 `--l2sp {0.3,1,3}` `--unfreeze {2,3}` `--epochs {15,25}`; Stage-2 `--wise-ft {0.3,0.5,0.7,0.9}` `--unfreeze {4,6,8}`. **Decide changes on an external new-center proxy (RARE25-val), not the same-center val below** — and only trust a change when its paired-Δ **AUROC/AUPRC** CI is clear of 0 (PPV@90R is too noisy at ~1% prevalence to rank configs).

## Val scoring → competition metric (PPV@90R @1% prevalence, bootstrap median + CI)
Scores `val_colab.csv` with the 3-seed ensemble and reports the leaderboard metric the same way the grader does (curve-point PPV@90R, 1% prevalence, bootstrap), plus AUROC/AUPRC. ⚠️ **This is SAME-CENTER (both centers were in training) → optimistic** (it read 0.65 vs the real new-center 0.018). Use it only as a smoke test that inference works; the honest new-center number comes from **RARE25-val / the leaderboard**, not here. Read the **CI**, not the point.

In [ ]:
# ---- score val with the 3-seed ensemble, then report the 5 trusted metrics ----
import os, csv, shutil, numpy as np
# restore ensemble from Drive if the runtime was reset
for s in [0, 1, 2]:
    if not os.path.exists(f'ship_seed{s}.pt') and os.path.exists(f'{DRIVE_DIR}/ship_seed{s}.pt'):
        shutil.copy(f'{DRIVE_DIR}/ship_seed{s}.pt', f'ship_seed{s}.pt')
assert all(os.path.exists(f'ship_seed{s}.pt') for s in [0, 1, 2]), 'ship_seed*.pt missing — run the ship cell or copy from Drive'
assert os.path.exists('val_colab.csv'), 'val_colab.csv missing — run the CSV-builder cell (needs out/val)'

!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --csv val_colab.csv --out preds.csv

from phase3 import evaluate as ev
# preds.csv = name,score,label ; pull center from val_colab.csv by frame name
center_by = {os.path.splitext(os.path.basename(r['path']))[0]: r.get('center', '')
             for r in csv.DictReader(open('val_colab.csv'))}
P = list(csv.DictReader(open('preds.csv')))
y = np.array([int(r['label']) for r in P]); s = np.array([float(r['score']) for r in P])
cen = np.array([center_by.get(r['name'], '') for r in P])
print(f'val frames={len(y)}  pos={int(y.sum())}  centers={sorted(set(cen))}\n')

hdr = f"{'split':10s} {'n':>4s} {'pos':>3s} | {'PPV@90R':>8s} {'CI_low':>7s} {'CI_high':>7s} | {'AUROC':>6s} {'AUPRC':>6s}"
print(hdr); print('-' * len(hdr))


def line(tag, yv, sv, cv):
    r = ev.report_full(yv, sv, cv if len(set(cv)) > 1 else None, target=0.9, prevalence=0.01, B=2000)
    print(f"{tag:10s} {r['n']:>4d} {r['pos']:>3d} | {r['ppv90']:>8.3f} {r['ci_lo']:>7.3f} {r['ci_hi']:>7.3f} | "
          f"{r['auroc']:>6.3f} {r['auprc']:>6.3f}")


line('POOLED', y, s, cen)
for cc in sorted(set(cen)):
    mk = cen == cc
    if mk.sum() and y[mk].sum() > 0:
        line(cc, y[mk], s[mk], cen[mk])
print("\nPPV@90R = curve-point @1% prevalence (leaderboard metric, HIGH variance at few pos — read the CI).")
print("AUROC/AUPRC = threshold-free ranking, STABLE at few pos, but NOT the 1%-operating-point score.")
print("SAME-CENTER val is optimistic (ship saw both centers); the honest new-center number is RARE25-val / the leaderboard.")